<a href="https://colab.research.google.com/github/shashikawka/268452H-CS5998-Capstone-Project/blob/main/02_data_preprocessing_and_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/CAPSTONE_PROJECT/MetroPT3.csv')

print("Shape:", df.shape)
display(df.head())

df.info()

Shape: (1516948, 17)


,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1516948 entries, 0 to 1516947
Data columns (total 17 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   Unnamed: 0       1516948 non-null  int64  
 1   timestamp        1516948 non-null  object 
 2   TP2              1516948 non-null  float64
 3   TP3              1516948 non-null  float64
 4   H1               1516948 non-null  float64
 5   DV_pressure      1516948 non-null  float64
 6   Reservoirs       1516948 non-null  float64
 7   Oil_temperature  1516948 non-null  float64
 8   Motor_current    1516948 non-null  float64
 9   COMP             1516948 non-null  float64
 10  DV_eletric       1516948 non-null  float64
 11  Towers           1516948 non-null  float64
 12  MPG              1516948 non-null  float64
 13  LPS              1516948 non-null  float64
 14  Pressure_switch  1516948 non-null  float64
 15  Oil_level        1516948 non-null  float64
 16  Caudal_impulses  1

In [3]:
# Remove the unnecessary  column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Ensure chronological order
df = df.sort_values('timestamp').reset_index(drop=True)

print("data after clean:", df.shape)
print("start:", df['timestamp'].min())
print("end:", df['timestamp'].max())

data after clean: (1516948, 16)
start: 2020-02-01 00:00:00
end: 2020-09-01 03:59:50


In [4]:
# Calculate time difference between consecutive observations
df['time_diff'] = df['timestamp'].diff()

# Mark a new segment whenever the gap is greater than 1 minute
df['new_segment'] = (
    df['time_diff'] > pd.Timedelta(minutes=1)
).astype(int)

# Assign a segment ID
df['segment_id'] = df['new_segment'].cumsum()

print("Number of continuous segments:", df['segment_id'].nunique())

Number of continuous segments: 332


##### 332 timestamp differences were greater than 1 minute.
Not deleting those rows or filling the missing periods. Instead,  mark where a new continuous segment begins.

In [5]:
#Define helper columns
#time_diff, new_segment, segment_id

segment_summary = df.groupby('segment_id').agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    observations=('timestamp', 'size')
)

print(segment_summary.head(10))
print("\nNumber of segments:", len(segment_summary))
print("Smallest segment:", segment_summary['observations'].min())
print("Largest segment:", segment_summary['observations'].max())

                    start_time            end_time  observations
segment_id                                                      
0          2020-02-01 00:00:00 2020-02-01 12:48:40          4654
1          2020-02-01 12:53:47 2020-02-01 19:40:04          2460
2          2020-02-01 23:15:33 2020-02-01 23:20:21            30
3          2020-02-02 04:34:27 2020-02-02 07:37:40          1110
4          2020-02-02 07:41:52 2020-02-05 00:32:21         23550
5          2020-02-05 00:43:00 2020-02-05 01:07:37           150
6          2020-02-05 04:28:42 2020-02-06 02:07:01          7860
7          2020-02-06 07:52:21 2020-02-06 14:08:53          2280
8          2020-02-06 14:19:23 2020-02-06 19:16:35          1800
9          2020-02-06 23:33:19 2020-02-06 23:48:02            90

Number of segments: 332
Smallest segment: 29
Largest segment: 36599


#####Temporal segmentation: Since the dataset contains occasional gaps in the timestamp sequence, gaps greater than one minute were used to separate the data into continuous segments. This resulted in 332 segments. The segmentation preserves all original observations while preventing future time-series windows from spanning large temporal gaps.

In [6]:
#chronological data split
failure_periods = [
    ('Failure 1', '2020-04-18 00:00:00', '2020-04-18 23:59:00'),
    ('Failure 2', '2020-05-29 23:30:00', '2020-05-30 06:00:00'),
    ('Failure 3', '2020-06-05 10:00:00', '2020-06-07 14:30:00'),
    ('Failure 4', '2020-07-15 14:30:00', '2020-07-15 19:00:00')
]

for name, start, end in failure_periods:
    print(name)
    print(" Start:", start)
    print(" End:  ", end)

Failure 1
 Start: 2020-04-18 00:00:00
 End:   2020-04-18 23:59:00
Failure 2
 Start: 2020-05-29 23:30:00
 End:   2020-05-30 06:00:00
Failure 3
 Start: 2020-06-05 10:00:00
 End:   2020-06-07 14:30:00
Failure 4
 Start: 2020-07-15 14:30:00
 End:   2020-07-15 19:00:00


#####should not randomly split rows. Split around the failure periods and chronological order,

In [7]:
#calculate exact positions of failures
for name, start, end in failure_periods:

    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    failure_rows = df[
        (df['timestamp'] >= start) &
        (df['timestamp'] <= end)
    ]

    print(
        name,
        "| rows:", len(failure_rows),
        "| start index:", failure_rows.index.min(),
        "| end index:", failure_rows.index.max()
    )

Failure 1 | rows: 8657 | start index: 562564 | end index: 571220
Failure 2 | rows: 2360 | start index: 840740 | end index: 843099
Failure 3 | rows: 17315 | start index: 890810 | end index: 908124
Failure 4 | rows: 1622 | start index: 1171093 | end index: 1172714


In [8]:
# Default: observation is outside a documented failure period
df['failure_period'] = 0

for name, start, end in failure_periods:
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    mask = (
        (df['timestamp'] >= start) &
        (df['timestamp'] <= end)
    )

    df.loc[mask, 'failure_period'] = 1

print(df['failure_period'].value_counts())

print(
    "Percentage within documented failure periods:",
    df['failure_period'].mean() * 100
)

failure_period
0    1486994
1      29954
Name: count, dtype: int64
Percentage within documented failure periods: 1.974622729322297


failure_period = 0
→ Outside a documented company failure interval

failure_period = 1
→ Inside a documented company failure interval

##Calculate chronological boundaries


#####Failure 1 → April 18
#####Failure 2 → May 29–30
#####Failure 3 → June 5–7
#####Failure 4 → July 15

In [9]:
n = len(df)

train_end_idx = int(n * 0.60) #find 60% position
val_end_idx = int(n * 0.80)

# 0% – 60%   → Train
# 60% – 80%  → Validation
# 80% – 100% → Test

print("Total observations:", n)
print("Train end index:", train_end_idx)
print("Validation end index:", val_end_idx)

print("\nTrain:")
print(df.iloc[0]['timestamp'], "to", df.iloc[train_end_idx - 1]['timestamp'])#timestamp from 1st raw and last raw from training set

print("\nValidation:")
print(df.iloc[train_end_idx]['timestamp'], "to", df.iloc[val_end_idx - 1]['timestamp'])

print("\nTest:")
print(df.iloc[val_end_idx]['timestamp'], "to", df.iloc[-1]['timestamp'])

Total observations: 1516948
Train end index: 910168
Validation end index: 1213558

Train:
2020-02-01 00:00:00 to 2020-06-08 17:58:17

Validation:
2020-06-08 17:58:27 to 2020-07-20 15:58:34

Test:
2020-07-20 15:58:44 to 2020-09-01 03:59:50


OBSERVATION : all four documented failures occur before the test set begins. The test set from July 20 to September 1 contains none of the four known failure intervals.

In [10]:
first_failure_start = pd.to_datetime('2020-04-18 00:00:00')

pre_failure_df = df[
    df['timestamp'] < first_failure_start
].copy()

evaluation_df = df[
    df['timestamp'] >= first_failure_start
].copy()

print("Pre-failure observations:", len(pre_failure_df))
print(
    "start:",
    pre_failure_df['timestamp'].min(),
    "to",
    pre_failure_df['timestamp'].max()
)

print("\nEvaluation observations:", len(evaluation_df))
print(
    "start:",
    evaluation_df['timestamp'].min(),
    "to",
    evaluation_df['timestamp'].max()
)

print("\nhow much propotion of the dataset before first failure:",
      len(pre_failure_df) / len(df) * 100)

Pre-failure observations: 562564
start: 2020-02-01 00:00:00 to 2020-04-17 23:59:49

Evaluation observations: 954384
start: 2020-04-18 00:00:01 to 2020-09-01 03:59:50

how much propotion of the dataset before first failure: 37.085252757510474


##START DATA SPLIT

In [11]:
#create the chronological train and validation sets from the 562,564 observations before Failure 1.
# Chronological 70/30 split of the pre-failure data

split_idx = int(len(pre_failure_df) * 0.70)

train_df = pre_failure_df.iloc[:split_idx].copy()
val_df = pre_failure_df.iloc[split_idx:].copy()

print("Training data:", len(train_df))
print("Validation data:", len(val_df))

print("\nTraining period:")
print(train_df['timestamp'].min(), "to", train_df['timestamp'].max())

print("\nValidation period:")
print(val_df['timestamp'].min(), "to", val_df['timestamp'].max())

print("\nTraining % of full dataset:",
      len(train_df) / len(df) * 100)

print("Validation % of full dataset:",
      len(val_df) / len(df) * 100)

print("Evaluation % of full dataset:",
      len(evaluation_df) / len(df) * 100)

Training data: 393794
Validation data: 168770

Training period:
2020-02-01 00:00:00 to 2020-03-25 00:00:26

Validation period:
2020-03-25 00:00:36 to 2020-04-17 23:59:49

Training % of full dataset: 25.959624192787096
Validation % of full dataset: 11.12562856472338
Evaluation % of full dataset: 62.91474724248952


#####train/validation boundary would probably fall inside the same segment, because the two timestamps were only 10 seconds apart.

But earlier, we had already divided the dataset into continuous segments whenever there was a gap > 1 minute.
So the question became:
 arbitrary 70% boundary cut through one continuous segment?

In [12]:
# Check segment IDs at the train-validation boundary

last_train_segment = train_df.iloc[-1]['segment_id']
first_val_segment = val_df.iloc[0]['segment_id']

print("Last training timestamp:", train_df.iloc[-1]['timestamp'])
print("First validation timestamp:", val_df.iloc[0]['timestamp'])

print("\nLast training segment ID:", last_train_segment)
print("First validation segment ID:", first_val_segment)

if last_train_segment == first_val_segment:
    print("\nThe train/validation boundary is inside the same continuous segment.")
else:
    print("\nThe train/validation boundary is already between two continuous segments.")

Last training timestamp: 2020-03-25 00:00:26
First validation timestamp: 2020-03-25 00:00:36

Last training segment ID: 74
First validation segment ID: 74

The train/validation boundary is inside the same continuous segment.


In [13]:
# Move train,validation boundary to the next continuous segment

boundary_segment = val_df.iloc[0]['segment_id']

# Training: everything before the next segment
train_df = pre_failure_df[
    pre_failure_df['segment_id'] <= boundary_segment
].copy()

# Validation: starts from the next segment
val_df = pre_failure_df[
    pre_failure_df['segment_id'] > boundary_segment
].copy()

print("Training observations:", len(train_df))
print("Validation observations:", len(val_df))

print("\nTraining period:")
print(train_df['timestamp'].min(), "to", train_df['timestamp'].max())

print("\nValidation period:")
print(val_df['timestamp'].min(), "to", val_df['timestamp'].max())

print("\nLast training segment:", train_df.iloc[-1]['segment_id'])
print("First validation segment:", val_df.iloc[0]['segment_id'])

print("\nGap between train and validation:",
      val_df.iloc[0]['timestamp'] - train_df.iloc[-1]['timestamp'])

Training observations: 423145
Validation observations: 139419

Training period:
2020-02-01 00:00:00 to 2020-03-28 08:49:39

Validation period:
2020-03-28 23:05:32 to 2020-04-17 23:59:49

Last training segment: 74
First validation segment: 75

Gap between train and validation: 0 days 14:15:53


CONCLUTION : Initially split the pre-failure data at 70%, discovered that this cut continuous segment 74 in half, and therefore moved the split to the natural gap between segments 74 and 75.

###Define 15 sensor features

In [14]:
sensor_features = [
    'TP2',
    'TP3',
    'H1',
    'DV_pressure',
    'Reservoirs',
    'Oil_temperature',
    'Motor_current',
    'COMP',
    'DV_eletric',
    'Towers',
    'MPG',
    'LPS',
    'Pressure_switch',
    'Oil_level',
    'Caudal_impulses'
]

print("Number of sensor features:", len(sensor_features))

print("\nTraining feature shape:",
      train_df[sensor_features].shape)

print("Validation feature shape:",
      val_df[sensor_features].shape)

print("Evaluation feature shape:",
      evaluation_df[sensor_features].shape)


Number of sensor features: 15

Training feature shape: (423145, 15)
Validation feature shape: (139419, 15)
Evaluation feature shape: (954384, 15)


In [15]:
#Separate continuous and digital features
continuous_features = [
    'TP2',
    'TP3',
    'H1',
    'DV_pressure',
    'Reservoirs',
    'Oil_temperature',
    'Motor_current'
]

digital_features = [
    'COMP',
    'DV_eletric',
    'Towers',
    'MPG',
    'LPS',
    'Pressure_switch',
    'Oil_level',
    'Caudal_impulses'
]

print("Continuous features:", len(continuous_features))
print("Digital features:", len(digital_features))

Continuous features: 7
Digital features: 8


###Scale the continuous sensors

#####Scale the 7 continuous features.
#####Keep the 8 digital features as 0/1.
#####Fit the scaler on training data only.
#####Apply the same scaler to validation and evaluation data.

In [16]:
from sklearn.preprocessing import StandardScaler

# Create copies to preserve the original datasets
train_scaled = train_df.copy()
val_scaled = val_df.copy()
evaluation_scaled = evaluation_df.copy()

# Initialize the scaler
scaler = StandardScaler()

# Learn mean and standard deviation from TRAINING data only
train_scaled[continuous_features] = scaler.fit_transform(
    train_df[continuous_features]
)

# Apply the SAME transformation to validation and evaluation
val_scaled[continuous_features] = scaler.transform(
    val_df[continuous_features]
)

evaluation_scaled[continuous_features] = scaler.transform(
    evaluation_df[continuous_features]
)

print("Scaling completed successfully.")

print("\nTraining shape:", train_scaled.shape)
print("Validation shape:", val_scaled.shape)
print("Evaluation shape:", evaluation_scaled.shape)

Scaling completed successfully.

Training shape: (423145, 20)
Validation shape: (139419, 20)
Evaluation shape: (954384, 20)


NOTE : use fit_transform() only on training data. Validation and evaluation use transform() because they must not influence the scaling parameters.

In [17]:
# Check continuous sensor statistics after scaling

print("Training continuous feature means:")
print(train_scaled[continuous_features].mean().round(3))

print("\nTraining continuous feature standard deviations:")
print(train_scaled[continuous_features].std(ddof=0).round(3))

print("\nDigital feature values:")
for col in digital_features:
    print(col, sorted(train_scaled[col].unique()))

Training continuous feature means:
TP2                0.0
TP3               -0.0
H1                 0.0
DV_pressure       -0.0
Reservoirs         0.0
Oil_temperature    0.0
Motor_current     -0.0
dtype: float64

Training continuous feature standard deviations:
TP2                1.0
TP3                1.0
H1                 1.0
DV_pressure        1.0
Reservoirs         1.0
Oil_temperature    1.0
Motor_current      1.0
dtype: float64

Digital feature values:
COMP [np.float64(0.0), np.float64(1.0)]
DV_eletric [np.float64(0.0), np.float64(1.0)]
Towers [np.float64(0.0), np.float64(1.0)]
MPG [np.float64(0.0), np.float64(1.0)]
LPS [np.float64(0.0), np.float64(1.0)]
Pressure_switch [np.float64(0.0), np.float64(1.0)]
Oil_level [np.float64(0.0), np.float64(1.0)]
Caudal_impulses [np.float64(0.0), np.float64(1.0)]



SUMMARY : Feature Scaling: 7 continuous sensor features were standardized using StandardScaler fitted exclusively on the training dataset.

The same scaling parameters were applied to validation and evaluation datasets to prevent data leakage.

Eight binary sensor features kept as it is in their original 0/1 representation.

Verification confirmed approximately zero means and unit standard deviations for all seven continuous training features.